# CATSA 2-Class Train + Eval (1D-CNN)

CNN-TF 노트북과 동일한 전처리/데이터 구성에서, 백엔드 모델은 1D-CNN을 사용합니다.

## Label Mapping (Train/Eval: 2-class)
- `0`: Baseline -> `Baseline`
- `1`: Stress -> (`Nback`, `Stroop`, `Sudoku`)

## Eval Mapping (Report: 2-class)
- `0`: Baseline
- `1`: Stress

## Inputs (Raw BVP branch 제거)
- ACC(32Hz): `(B, 3, 1920)`
- Slow group(4Hz): `(B, 5, 240)` where channels = `[EDA_tonic, EDA_phasic, TEMP, HR, HRV]`
- BVP는 HR/HRV 추출용으로만 사용, 원시 파형은 모델 입력에서 제외

## Fusion + 1D-CNN
- ACC branch -> `(B, 32, 240)`
- Slow branch -> `(B, 16, 240)`
- concat -> `(B, 48, 240)`
- Temporal CNN blocks -> Global average pooling -> FC(2)

## Train Stride (고정)
- Train: `20s`
- Eval(Val/Test): `60s`

## Subject-wise Baseline Normalization
- 각 피험자 `Baseline` 구간의 평균/표준편차로 해당 피험자 모든 태스크를 정규화

## Training Policy
- Weighted sampler off
- Class-weighted CrossEntropy on (auto class weights from train windows)
- Manual class weights off
- Focal Loss off
- Early stopping monitor: `val loss (2-class CE)`


In [3]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.signal import find_peaks, butter, filtfilt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ===================== User Config =====================
DATASET_ROOT = Path('/home/binghin2/Myproject/Dataset/CATSA')

# Train as 2 classes (Baseline vs Stress)
TASK_TO_CLASS = {
    'Baseline': 0,
    'Nback': 1,
    'Stroop': 1,
    'Sudoku': 1,
}
CLASS_NAMES = ['Baseline', 'Stress']

WINDOW_SEC = 60
TRAIN_STRIDE_SEC = 20
STRIDE_SEC_EVAL = 60

# Native sampling rates
FS_BVP = 64
FS_ACC = 32
FS_SLOW = 4   # EDA/TEMP and derived HR/HRV

# 60 sec expected lengths
LEN_ACC = WINDOW_SEC * FS_ACC   # 1920
LEN_SLOW = WINDOW_SEC * FS_SLOW # 240

BATCH_SIZE = 32
EPOCHS = 40
LR = 3e-4
WEIGHT_DECAY = 3e-4
SEED = 42

# Overfitting control
DROPOUT = 0.5
EARLY_STOP_PATIENCE = 10
EARLY_STOP_LOSS_MIN_DELTA = 1e-3
LABEL_SMOOTHING = 0.05

# Preset switch
EXPERIMENT_PRESET = 'val_loss_min'  # options: 'val_loss_min', 'deepflow_recall_boost'
PRESET_CONFIGS = {
    'val_loss_min': {
        'lr': 2e-4,
        'weight_decay': 4e-4,
        'dropout': 0.5,
        'label_smoothing': 0.05,
        'early_stop_patience': 12,
        'early_stop_loss_min_delta': 1e-3,
        'plateau_factor': 0.3,
        'plateau_patience': 1,
        'min_lr': 1e-6,
        'use_class_weights': True,
    },
    'deepflow_recall_boost': {
        'lr': 2e-4,
        'weight_decay': 3e-4,
        'dropout': 0.45,
        'label_smoothing': 0.02,
        'early_stop_patience': 12,
        'early_stop_loss_min_delta': 8e-4,
        'plateau_factor': 0.5,
        'plateau_patience': 2,
        'min_lr': 1e-6,
        'use_class_weights': True,
    },
}
if EXPERIMENT_PRESET not in PRESET_CONFIGS:
    raise ValueError(f'Unknown EXPERIMENT_PRESET: {EXPERIMENT_PRESET}')
cfg = PRESET_CONFIGS[EXPERIMENT_PRESET]
LR = cfg['lr']
WEIGHT_DECAY = cfg['weight_decay']
DROPOUT = cfg['dropout']
LABEL_SMOOTHING = cfg['label_smoothing']
EARLY_STOP_PATIENCE = cfg['early_stop_patience']
EARLY_STOP_LOSS_MIN_DELTA = cfg['early_stop_loss_min_delta']
PLATEAU_FACTOR = cfg['plateau_factor']
PLATEAU_PATIENCE = cfg['plateau_patience']
MIN_LR = cfg['min_lr']
USE_CLASS_WEIGHTS = cfg['use_class_weights']

# Imbalance controls
USE_WEIGHTED_SAMPLER = False

# Manual class weights (Baseline, Stress)
USE_MANUAL_CLASS_WEIGHTS = False
MANUAL_CLASS_WEIGHTS = [1.0, 1.0]

# Loss controls
USE_FOCAL_LOSS = False
FOCAL_GAMMA = 2.0

# LSTM config (downsized)
LSTM_HIDDEN = 32
LSTM_LAYERS = 1
LSTM_BIDIRECTIONAL = True

# EDA decomposition
EDA_TONIC_CUTOFF_HZ = 0.05
EDA_FILTER_ORDER = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print('Device:', DEVICE)
print('Dataset root:', DATASET_ROOT)
print('Experiment preset:', EXPERIMENT_PRESET)
print('Dropout:', DROPOUT)
print('Optimizer:', {'lr': LR, 'weight_decay': WEIGHT_DECAY})
print('LR scheduler:', {'type': 'ReduceLROnPlateau', 'factor': PLATEAU_FACTOR, 'patience': PLATEAU_PATIENCE, 'min_lr': MIN_LR})
print('Label smoothing:', LABEL_SMOOTHING)
print('Early stopping monitor: val_loss', {'patience': EARLY_STOP_PATIENCE, 'min_delta': EARLY_STOP_LOSS_MIN_DELTA})
print('Train stride (sec):', TRAIN_STRIDE_SEC)
print('Model backend: 1D-CNN')
print('Weighted sampler:', USE_WEIGHTED_SAMPLER)
print('Use class weights:', USE_CLASS_WEIGHTS)
print('Use manual class weights:', USE_MANUAL_CLASS_WEIGHTS, '| values:', MANUAL_CLASS_WEIGHTS)
print('Use focal loss:', USE_FOCAL_LOSS, '| gamma:', FOCAL_GAMMA)
print('Train classes:', CLASS_NAMES)
print('Eval classes:', CLASS_NAMES)


Device: cuda
Dataset root: /home/binghin2/Myproject/Dataset/CATSA
Experiment preset: val_loss_min
Dropout: 0.5
Optimizer: {'lr': 0.0002, 'weight_decay': 0.0004}
LR scheduler: {'type': 'ReduceLROnPlateau', 'factor': 0.3, 'patience': 1, 'min_lr': 1e-06}
Label smoothing: 0.05
Early stopping monitor: val_loss {'patience': 12, 'min_delta': 0.001}
Train stride (sec): 20
Model backend: 1D-CNN
Weighted sampler: False
Use class weights: True
Use manual class weights: False | values: [1.0, 1.0]
Use focal loss: False | gamma: 2.0
Train classes: ['Baseline', 'Stress']
Eval classes: ['Baseline', 'Stress']


In [4]:
def read_csv_array(path: Path) -> np.ndarray:
    return pd.read_csv(path).values.astype(np.float32)


def bvp_to_hr_hrv_4hz(bvp_64: np.ndarray, fs: int = 64, out_fs: int = 4,
                     min_peak_distance_sec: float = 0.30,
                     hrv_window_sec: int = 10):
    sig = np.asarray(bvp_64, dtype=np.float32).reshape(-1)
    min_dist = max(1, int(fs * min_peak_distance_sec))
    peaks, _ = find_peaks(sig, distance=min_dist)

    n4 = len(sig) // (fs // out_fs)
    t4 = np.arange(n4) / float(out_fs)

    if len(peaks) < 3:
        return np.zeros(n4, dtype=np.float32), np.zeros(n4, dtype=np.float32)

    t_peaks = peaks / float(fs)
    ibi = np.diff(t_peaks)
    ibi = np.clip(ibi, 1e-3, None)
    hr = 60.0 / ibi

    t_ibi = t_peaks[1:]
    hr_4 = np.interp(t4, t_ibi, hr).astype(np.float32)
    ibi_series = pd.Series(ibi)
    hrv = ibi_series.rolling(window=max(2, int(hrv_window_sec)), min_periods=1).std().fillna(0).values
    hrv_4 = np.interp(t4, t_ibi, hrv).astype(np.float32)

    return hr_4, hrv_4


def decompose_eda(eda_4hz: np.ndarray, fs: int = 4, cutoff_hz: float = 0.05, order: int = 2):
    x = np.asarray(eda_4hz, dtype=np.float32).reshape(-1)
    if len(x) < (order * 6 + 1):
        tonic = x.copy()
        phasic = np.zeros_like(x)
        return tonic, phasic

    nyq = 0.5 * fs
    norm = max(1e-4, min(cutoff_hz / nyq, 0.99))
    b, a = butter(order, norm, btype='low')
    tonic = filtfilt(b, a, x).astype(np.float32)
    phasic = (x - tonic).astype(np.float32)
    return tonic, phasic


def load_task_modalities(subject_dir: Path, task: str):
    task_dir = subject_dir / task
    paths = {
        'acc': task_dir / 'ACC.csv',
        'bvp': task_dir / 'BVP.csv',
        'eda': task_dir / 'EDA.csv',
        'temp': task_dir / 'TEMP.csv',
    }
    if not all(p.exists() for p in paths.values()):
        return None

    acc = read_csv_array(paths['acc'])
    bvp = read_csv_array(paths['bvp']).reshape(-1)
    eda = read_csv_array(paths['eda']).reshape(-1)
    temp = read_csv_array(paths['temp']).reshape(-1)

    if acc.ndim == 1:
        acc = acc.reshape(-1, 1)
    if acc.shape[1] < 3:
        acc = np.tile(acc, (1, 3))[:, :3]
    else:
        acc = acc[:, :3]

    hr_4, hrv_4 = bvp_to_hr_hrv_4hz(bvp, fs=FS_BVP, out_fs=FS_SLOW)
    scl_4, scr_4 = decompose_eda(
        eda, fs=FS_SLOW,
        cutoff_hz=EDA_TONIC_CUTOFF_HZ,
        order=EDA_FILTER_ORDER,
    )

    t4_from_acc = len(acc) // (FS_ACC // FS_SLOW)
    t4_from_bvp = len(bvp) // (FS_BVP // FS_SLOW)
    t4 = min(len(scl_4), len(scr_4), len(temp), len(hr_4), len(hrv_4), t4_from_acc, t4_from_bvp)
    if t4 < LEN_SLOW:
        return None

    scl_4 = scl_4[:t4]
    scr_4 = scr_4[:t4]
    temp = temp[:t4]
    hr_4 = hr_4[:t4]
    hrv_4 = hrv_4[:t4]
    acc = acc[: t4 * (FS_ACC // FS_SLOW)]

    slow = np.stack([scl_4, scr_4, temp, hr_4, hrv_4], axis=1).astype(np.float32)

    return {
        'acc': acc.astype(np.float32),
        'slow': slow,
    }


def compute_subject_baseline_stats(subject_dir: Path):
    """
    Per-subject normalization stats from Baseline only.
    Returns dict with per-channel mean/std for acc and slow.
    """
    baseline = load_task_modalities(subject_dir, 'Baseline')
    if baseline is None:
        return None

    acc = baseline['acc']   # (T,3)
    slow = baseline['slow'] # (T,5)

    acc_mean = acc.mean(axis=0, keepdims=True)
    acc_std = acc.std(axis=0, keepdims=True) + 1e-6
    slow_mean = slow.mean(axis=0, keepdims=True)
    slow_std = slow.std(axis=0, keepdims=True) + 1e-6

    return {
        'acc_mean': acc_mean.astype(np.float32),
        'acc_std': acc_std.astype(np.float32),
        'slow_mean': slow_mean.astype(np.float32),
        'slow_std': slow_std.astype(np.float32),
    }


def apply_subject_baseline_norm(mods: dict, stats: dict):
    acc = (mods['acc'] - stats['acc_mean']) / stats['acc_std']
    slow = (mods['slow'] - stats['slow_mean']) / stats['slow_std']
    return {
        'acc': acc.astype(np.float32),
        'slow': slow.astype(np.float32),
    }


def create_windows(mods: dict, label: int, stride_sec: int):
    stride4 = stride_sec * FS_SLOW
    w4 = LEN_SLOW
    w32 = LEN_ACC
    ratio32 = FS_ACC // FS_SLOW

    n4 = len(mods['slow'])
    out = []
    start4 = 0
    while start4 + w4 <= n4:
        s32 = start4 * ratio32

        slow_win = mods['slow'][start4:start4 + w4]
        acc_win = mods['acc'][s32:s32 + w32]

        if len(slow_win) != w4 or len(acc_win) != w32:
            break

        out.append({
            'slow': slow_win.T.copy(),
            'acc': acc_win.T.copy(),
            'y': int(label),
        })
        start4 += stride4

    return out


def list_subjects(root: Path):
    return sorted([p.name for p in root.glob('Sub*') if p.is_dir()])


def split_subjects(subjects, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(subjects))
    n = len(subjects)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]

    train_subjects = [subjects[i] for i in train_idx]
    val_subjects = [subjects[i] for i in val_idx]
    test_subjects = [subjects[i] for i in test_idx]
    return train_subjects, val_subjects, test_subjects


def build_split_windows(subjects, is_train=True):
    all_windows = []
    skipped_no_baseline = 0

    for s in subjects:
        sdir = DATASET_ROOT / s
        stats = compute_subject_baseline_stats(sdir)
        if stats is None:
            skipped_no_baseline += 1
            continue

        for task, cls in TASK_TO_CLASS.items():
            mods = load_task_modalities(sdir, task)
            if mods is None:
                continue

            mods = apply_subject_baseline_norm(mods, stats)
            stride = TRAIN_STRIDE_SEC if is_train else STRIDE_SEC_EVAL
            all_windows.extend(create_windows(mods, cls, stride_sec=stride))

    if skipped_no_baseline > 0:
        print(f'Skipped subjects (missing usable Baseline): {skipped_no_baseline}')

    return all_windows


subjects = list_subjects(DATASET_ROOT)
tr_subj, va_subj, te_subj = split_subjects(subjects, seed=SEED)
print(f'Subject split -> train:{len(tr_subj)} val:{len(va_subj)} test:{len(te_subj)}')


Subject split -> train:35 val:7 test:8


In [5]:
class CATSAMultiModalDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        return (
            torch.tensor(w['acc'], dtype=torch.float32),
            torch.tensor(w['slow'], dtype=torch.float32),
            torch.tensor(w['y'], dtype=torch.long),
        )


class MultiBranchPureCNN(nn.Module):
    """
    Input:
      acc  = (B, 3, 1920)
      slow = (B, 5, 240)  [SCL, SCR, TEMP, HR, HRV]

    Fusion:
      acc_f  -> (B, 32, 240)
      slow_f -> (B, 16, 240)
      cat -> (B, 48, 240)
      Temporal CNN -> GAP -> FC(n_classes)
    """
    def __init__(self, n_classes=4, dropout=0.4):
        super().__init__()

        self.acc_branch = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=9, stride=4, padding=4),
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.slow_branch = nn.Sequential(
            nn.Conv1d(5, 16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(16, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.temporal_head = nn.Sequential(
            nn.Conv1d(48, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Linear(64, n_classes)

    def forward(self, acc, slow, return_features=False):
        acc_f = self.acc_branch(acc)                 # (B,32,240)
        slow_f = self.slow_branch(slow)              # (B,16,240)
        fused = torch.cat([acc_f, slow_f], dim=1)    # (B,48,240)

        feat = self.temporal_head(fused).squeeze(-1) # (B,64)
        logits = self.classifier(feat)

        if return_features:
            return logits, fused
        return logits


with torch.no_grad():
    m = MultiBranchPureCNN(
        n_classes=len(CLASS_NAMES),
        dropout=DROPOUT,
    ).to(DEVICE)
    acc = torch.randn(2, 3, LEN_ACC, device=DEVICE)
    slow = torch.randn(2, 5, LEN_SLOW, device=DEVICE)
    logits, fused = m(acc, slow, return_features=True)
    print('Expected fused feature shape: (2, 48, 240)')
    print('Actual fused feature shape  :', tuple(fused.shape))
    print('Logits shape                :', tuple(logits.shape))


Expected fused feature shape: (2, 48, 240)
Actual fused feature shape  : (2, 48, 240)
Logits shape                : (2, 2)


In [6]:
def class_weights_from_windows(train_windows, n_classes):
    y = np.array([w['y'] for w in train_windows], dtype=np.int64)
    cnt = np.bincount(y, minlength=n_classes).astype(np.float32)
    total = cnt.sum()
    w = total / (n_classes * np.maximum(cnt, 1.0))
    return torch.tensor(w, dtype=torch.float32)


def build_weighted_sampler(train_windows, n_classes):
    y = np.array([w['y'] for w in train_windows], dtype=np.int64)
    cnt = np.bincount(y, minlength=n_classes).astype(np.float32)
    class_w = 1.0 / np.maximum(cnt, 1.0)
    sample_w = class_w[y]
    sample_w = torch.tensor(sample_w, dtype=torch.double)
    return WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce)
        focal = ((1.0 - pt) ** self.gamma) * ce
        return focal.mean()


def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_pred, all_true = [], []

    with torch.no_grad():
        for acc, slow, y in loader:
            acc = acc.to(device)
            slow = slow.to(device)
            y = y.to(device)

            logits = model(acc, slow)
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0)
            pred = logits.argmax(dim=1)
            all_pred.extend(pred.cpu().numpy().tolist())
            all_true.extend(y.cpu().numpy().tolist())

    n = max(len(all_true), 1)
    acc = float(np.mean(np.array(all_pred) == np.array(all_true))) if len(all_true) > 0 else 0.0
    return total_loss / n, acc, all_pred, all_true


def map_labels_for_eval(labels):
    return [int(y) for y in labels]


def print_class_counts(windows, title, class_names):
    y = np.array([w['y'] for w in windows], dtype=np.int64)
    cnt = np.bincount(y, minlength=len(class_names))
    msg = ', '.join([f"{class_names[i]}:{int(cnt[i])}" for i in range(len(class_names))])
    print(f"{title}: {msg}")


def run_training():
    n_classes = len(CLASS_NAMES)

    print('Building windows with subject-wise baseline normalization...')
    tr_win = build_split_windows(tr_subj, is_train=True)
    va_win = build_split_windows(va_subj, is_train=False)
    te_win = build_split_windows(te_subj, is_train=False)

    print(f'Windows -> train:{len(tr_win)} val:{len(va_win)} test:{len(te_win)}')
    print_class_counts(tr_win, 'Train class windows (2-class)', CLASS_NAMES)
    print_class_counts(va_win, 'Val class windows (2-class)', CLASS_NAMES)
    print_class_counts(te_win, 'Test class windows (2-class)', CLASS_NAMES)

    if len(tr_win) == 0 or len(va_win) == 0 or len(te_win) == 0:
        raise RuntimeError('One of dataset splits has zero windows. Check dataset path and files.')

    train_ds = CATSAMultiModalDataset(tr_win)
    val_ds = CATSAMultiModalDataset(va_win)
    test_ds = CATSAMultiModalDataset(te_win)

    if USE_WEIGHTED_SAMPLER:
        sampler = build_weighted_sampler(tr_win, n_classes=n_classes)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = MultiBranchPureCNN(
        n_classes=n_classes,
        dropout=DROPOUT,
    ).to(DEVICE)

    class_w = class_weights_from_windows(tr_win, n_classes=n_classes).to(DEVICE)
    if USE_CLASS_WEIGHTS and USE_MANUAL_CLASS_WEIGHTS:
        if len(MANUAL_CLASS_WEIGHTS) != n_classes:
            raise ValueError('MANUAL_CLASS_WEIGHTS length must match number of classes.')
        class_w = torch.tensor(MANUAL_CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)
    print('Class weights:', [round(x, 4) for x in class_w.detach().cpu().tolist()])

    if USE_FOCAL_LOSS:
        alpha = class_w / class_w.mean()
        criterion = FocalLoss(alpha=alpha, gamma=FOCAL_GAMMA)
    elif USE_CLASS_WEIGHTS:
        criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=LABEL_SMOOTHING)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=PLATEAU_FACTOR,
        patience=PLATEAU_PATIENCE,
        min_lr=MIN_LR,
    )

    best_state = None
    best_val_loss = float('inf')
    best_val_acc = 0.0
    best_val_macro_f1 = 0.0
    patience_counter = 0

    print('\nEpoch | TrainLoss TrainAcc | ValLoss ValAcc ValF1(binary) | ES')
    print('-' * 80)

    for ep in range(1, EPOCHS + 1):
        model.train()
        tr_loss_sum = 0.0
        tr_true, tr_pred = [], []

        for acc, slow, y in train_loader:
            acc = acc.to(DEVICE)
            slow = slow.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(acc, slow)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            tr_loss_sum += loss.item() * y.size(0)
            tr_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
            tr_true.extend(y.detach().cpu().numpy().tolist())

        tr_n = max(len(tr_true), 1)
        tr_loss = tr_loss_sum / tr_n
        tr_acc = float(np.mean(np.array(tr_pred) == np.array(tr_true)))

        va_loss, va_acc, va_pred, va_true = eval_model(model, val_loader, criterion, DEVICE)
        scheduler.step(va_loss)
        va_true_bin = map_labels_for_eval(va_true)
        va_pred_bin = map_labels_for_eval(va_pred)
        va_macro_f1 = f1_score(va_true_bin, va_pred_bin, average='macro', zero_division=0)

        improved = va_loss < (best_val_loss - EARLY_STOP_LOSS_MIN_DELTA)
        if improved:
            best_val_macro_f1 = va_macro_f1
            best_val_loss = va_loss
            best_val_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            mark = ' *'
        else:
            patience_counter += 1
            mark = ''

        print(f'{ep:5d} | {tr_loss:8.4f} {tr_acc:8.2%} | {va_loss:7.4f} {va_acc:7.2%} {va_macro_f1:13.3f} | {patience_counter:2d}/{EARLY_STOP_PATIENCE}{mark}')

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'Early stopping triggered at epoch {ep} (best val loss: {best_val_loss:.4f})')
            break

    print('\nBest val macro-F1:', f'{best_val_macro_f1:.3f}')
    print('Best val loss           :', f'{best_val_loss:.4f}')
    print('Best val acc (2-class)  :', f'{best_val_acc:.2%}')
    if best_state is not None:
        model.load_state_dict(best_state)

    te_loss, te_acc, te_pred, te_true = eval_model(model, test_loader, criterion, DEVICE)
    te_macro_f1 = f1_score(te_true, te_pred, average='macro', zero_division=0)
    print('Test loss (2-class):', f'{te_loss:.4f}')
    print('Test acc  (2-class):', f'{te_acc:.2%}')
    print('Test macro-F1 (2-class):', f'{te_macro_f1:.3f}')

    print('\nClassification Report (Test, 2-class)')
    print(classification_report(te_true, te_pred, target_names=CLASS_NAMES, zero_division=0))

    cm2 = confusion_matrix(te_true, te_pred, labels=[0, 1])
    cm2_df = pd.DataFrame(cm2, index=[f'True_{c}' for c in CLASS_NAMES], columns=[f'Pred_{c}' for c in CLASS_NAMES])
    print('Confusion Matrix (Test, 2-class)')
    display(cm2_df)

    return model


model = run_training()


Building windows with subject-wise baseline normalization...
Windows -> train:980 val:84 test:96
Train class windows (2-class): Baseline:245, Stress:735
Val class windows (2-class): Baseline:21, Stress:63
Test class windows (2-class): Baseline:24, Stress:72
Class weights: [2.0, 0.6667]

Epoch | TrainLoss TrainAcc | ValLoss ValAcc ValF1(binary) | ES
--------------------------------------------------------------------------------
    1 |   0.6425   53.57% |  0.7104  25.00%         0.200 |  0/12 *
    2 |   0.5657   59.59% |  0.7644  28.57%         0.251 |  1/12
    3 |   0.4648   73.88% |  0.7337  41.67%         0.413 |  2/12
    4 |   0.4099   80.71% |  0.7225  46.43%         0.464 |  3/12
    5 |   0.4033   81.12% |  0.6980  50.00%         0.500 |  0/12 *
    6 |   0.3845   83.67% |  0.6529  52.38%         0.524 |  0/12 *
    7 |   0.3809   84.80% |  0.7018  51.19%         0.512 |  1/12
    8 |   0.3727   85.51% |  0.6163  54.76%         0.547 |  0/12 *
    9 |   0.3686   86.22% |  0.6

,Pred_Baseline,Pred_Stress
True_Baseline,24,0
True_Stress,33,39


In [7]:
# Save trained model checkpoint
from pathlib import Path
from datetime import datetime
import json

save_dir = Path('/home/binghin2/Myproject/Research/CATSA/Train/CNN/Save_model')
save_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
ckpt_path = save_dir / f'cnn_1dcnn_binary_baseline_stress_nobvp_{ts}.pth'
meta_path = save_dir / f'cnn_1dcnn_binary_baseline_stress_nobvp_{ts}_meta.json'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'model_type': 'MultiBranchPureCNN',
    'class_names': CLASS_NAMES,
    'task_to_class': TASK_TO_CLASS,
    'window_sec': WINDOW_SEC,
    'train_stride_sec': TRAIN_STRIDE_SEC,
    'stride_sec_eval': STRIDE_SEC_EVAL,
    'normalization': 'subject_baseline_zscore',
    'baseline_task': 'Baseline',
    'slow_channels': ['EDA_tonic', 'EDA_phasic', 'TEMP', 'HR', 'HRV'],
    'sampling_rates': {
        'acc': FS_ACC,
        'slow': FS_SLOW,
        'bvp_source_for_hr_hrv': FS_BVP,
    },
    'lengths': {
        'acc': LEN_ACC,
        'slow': LEN_SLOW,
    },
    'optimizer': {
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
    },
    'experiment_preset': EXPERIMENT_PRESET,
    'scheduler': {
        'name': 'ReduceLROnPlateau',
        'factor': PLATEAU_FACTOR,
        'patience': PLATEAU_PATIENCE,
        'min_lr': MIN_LR,
    },
    'imbalance_controls': {
        'use_weighted_sampler': USE_WEIGHTED_SAMPLER,
        'use_class_weights': USE_CLASS_WEIGHTS,
        'use_manual_class_weights': USE_MANUAL_CLASS_WEIGHTS,
        'manual_class_weights': MANUAL_CLASS_WEIGHTS,
    },
    'loss': {
        'use_focal_loss': USE_FOCAL_LOSS,
        'focal_gamma': FOCAL_GAMMA,
        'label_smoothing': LABEL_SMOOTHING,
    },
    'dropout': DROPOUT,
    'early_stopping': {
        'patience': EARLY_STOP_PATIENCE,
        'min_delta': EARLY_STOP_LOSS_MIN_DELTA,
        'monitor': 'val_loss',
    },
    'raw_bvp_branch_used': False,
}

torch.save(checkpoint, ckpt_path)

with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({
        'checkpoint_path': str(ckpt_path),
        'created_at': ts,
        'device': str(DEVICE),
        'model_type': 'MultiBranchPureCNN',
        'class_names': CLASS_NAMES,
        'train_stride_sec': TRAIN_STRIDE_SEC,
        'experiment_preset': EXPERIMENT_PRESET,
        'normalization': 'subject_baseline_zscore',
        'baseline_task': 'Baseline',
        'slow_channels': ['EDA_tonic', 'EDA_phasic', 'TEMP', 'HR', 'HRV'],
        'raw_bvp_branch_used': False,
        'use_manual_class_weights': USE_MANUAL_CLASS_WEIGHTS,
        'manual_class_weights': MANUAL_CLASS_WEIGHTS,
        'label_smoothing': LABEL_SMOOTHING,
    }, f, ensure_ascii=False, indent=2)

print('Saved checkpoint:', ckpt_path)
print('Saved metadata  :', meta_path)


Saved checkpoint: /home/binghin2/Myproject/Research/CATSA/Train/CNN/Save_model/cnn_1dcnn_binary_baseline_stress_nobvp_20260322_221522.pth
Saved metadata  : /home/binghin2/Myproject/Research/CATSA/Train/CNN/Save_model/cnn_1dcnn_binary_baseline_stress_nobvp_20260322_221522_meta.json
